# Experiment: Synthetic Neuro Validation

This notebook validates the active `nethobench` neuro benchmark on synthetic neural datasets with known structure. It estimates empirical oracle ceilings from independent draws of the same generator and then applies targeted perturbations that are designed to affect specific score families.

## Validation Logic

The notebook follows three steps.

1. Generate a reference synthetic dataset from a fixed latent dynamical system with calcium-like AR(1) observations.
2. Estimate an empirical ceiling by comparing the reference dataset against independent samples from the same generator.
3. Apply targeted perturbations in generator parameter space and test whether the intended metric family drops more than off-target families.

This is stronger than generic noise injection because the manipulated structure is known by construction.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Image, display

repo_root = Path.cwd().resolve()
for candidate in [repo_root, *repo_root.parents, Path.home() / 'nethobench', Path.home() / 'Desktop' / 'nethobench']:
    candidate = candidate.resolve()
    if (candidate / 'nethobench' / '__init__.py').is_file():
        repo_root = candidate
        break
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from nethobench.synthetic import SyntheticNeuralSpec, run_synthetic_neuro_validation

repo_root

## Experiment Configuration

The generator is intentionally simple and interpretable: a low-dimensional latent system with controllable decay, oscillatory drive, region loadings, and calcium smoothing. The defaults are sized to run quickly while still exercising all active neuro metrics.

In [ ]:
output_root = repo_root / 'outputs' / 'synthetic_validation_notebook'

spec = SyntheticNeuralSpec(
    n_sequences=12,
    seq_length=240,
    n_regions=16,
    latent_dim=4,
    burn_in=64,
    system_seed=11,
)

oracle_replicates = 3
save_datasets = True

print(spec)
output_root

## Run Synthetic Validation

This executes the public synthetic-validation pipeline and writes tables, figures, and CSV datasets under the chosen output directory.

In [ ]:
result = run_synthetic_neuro_validation(
    output_root=output_root,
    spec=spec,
    oracle_replicates=oracle_replicates,
    save_datasets=save_datasets,
)

sorted(result.keys())

## Oracle Ceilings

The oracle summaries report the benchmark response when both datasets come from the same underlying generator. These values are empirical ceilings rather than idealized scores of exactly `1.0`.

In [ ]:
family_oracle = pd.read_csv(result['family_oracle_summary_path'])
metric_oracle = pd.read_csv(result['metric_oracle_summary_path'])

display(family_oracle.sort_values('score_name').reset_index(drop=True))
display(metric_oracle.sort_values('mean', ascending=False).reset_index(drop=True))

## Targeted Selectivity

The selectivity tables measure relative drop from the oracle ceiling at the maximum perturbation magnitude. For a well-calibrated benchmark, the target family should usually show one of the largest drops for its corresponding perturbation.

In [ ]:
family_selectivity = pd.read_csv(result['family_selectivity_path'])
metric_selectivity = pd.read_csv(result['metric_selectivity_path'])

display(family_selectivity)
display(metric_selectivity)

## Figures

The pipeline saves three paper-style figure summaries: family ceilings versus targeted mismatches, family selectivity, metric selectivity, and family dose-response curves.

In [ ]:
for key in ['ceiling_plot', 'family_selectivity_plot', 'metric_selectivity_plot', 'dose_plot']:
    print(f'\n{key}: {result[key]}')
    display(Image(filename=str(result[key])))

## Metric Drilldown

This view extracts the strongest target-specific drops at the maximum perturbation magnitude so it is easy to inspect which individual metrics are most responsive within each family.

In [ ]:
rows = []
for _, row in metric_selectivity.iterrows():
    payload = row.drop(labels=['perturbation_name', 'target_family']).sort_values(ascending=False)
    top = payload.head(5)
    for metric_name, value in top.items():
        rows.append({
            'perturbation_name': row['perturbation_name'],
            'target_family': row['target_family'],
            'metric_name': metric_name,
            'relative_drop': value,
        })

top_metric_drops = pd.DataFrame(rows)
display(top_metric_drops)

## Exported Datasets

The generated CSVs include `sequenceId` and `itemPosition`, so they are immediately usable for downstream tooling such as Sequifier preprocessing or manual benchmark experiments.

In [ ]:
datasets_dir = result['output_root'] / 'datasets'
dataset_files = sorted(path.name for path in datasets_dir.glob('*.csv'))
pd.DataFrame({'dataset_file': dataset_files})